<a href="https://colab.research.google.com/github/syaharah/Proyek-Data-Wrangling/blob/main/Tugas_Individu_2_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas 2: Data Preparation
**Nama:** [Nama Mahasiswa]  
**Dataset:** data_tugas_2.csv — Dataset Kendaraan  
**Deskripsi:** Melakukan proses data cleaning dan normalisasi data sebagai bagian dari tahapan data preprocessing.

## Import Library & Load Dataset

In [7]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/content/sample_data/data_tugas 2.csv')

# Ganti tanda '?' dengan NaN (nilai kosong)
# '?' digunakan dalam dataset ini sebagai penanda nilai yang tidak tersedia
df.replace('?', np.nan, inplace=True)

# Konversi kolom numerik ke tipe float
# Diperlukan karena sebelumnya bertipe object akibat adanya '?'
numeric_cols = ['wheel-bases', 'length', 'width', 'height']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Tampilkan 5 baris pertama dataset
print('=== Preview Dataset ===')
print(df.head())
print(f'\nUkuran Dataset: {df.shape[0]} baris x {df.shape[1]} kolom')

=== Preview Dataset ===
  normalized-losses         make bahan-bakar jumlah-pintu   body-style  \
0               NaN  alfa-romero         gas          two  convertible   
1               NaN  alfa-romero         gas          two  convertible   
2               NaN  alfa-romero         gas          two    hatchback   
3               164         audi         gas         four        sedan   
4               164         audi         gas         four        sedan   

   wheel-bases  length  width  height  
0         88.6   168.8   64.1    48.8  
1         88.6   168.8   64.1    48.8  
2         94.5   171.2   65.5    52.4  
3         99.8   176.6   66.2    54.3  
4         99.4   176.6   66.4    54.3  

Ukuran Dataset: 205 baris x 9 kolom


---
## 1. Menampilkan Jumlah Data Hilang (Missing Values)
Langkah pertama adalah memeriksa dataset untuk mengetahui jumlah data hilang pada setiap variabel. Hal ini penting agar kita dapat menentukan strategi penanganan yang tepat.

In [8]:
# Menghitung jumlah missing value per kolom
missing_count = df.isnull().sum()

# Menghitung persentase missing value per kolom terhadap total baris
missing_percent = (df.isnull().sum() / len(df)) * 100

# Membuat tabel ringkasan missing value
missing_summary = pd.DataFrame({
    'Jumlah Missing Value': missing_count,
    'Persentase (%)': missing_percent.round(2)
})

# Urutkan dari yang paling banyak missing value
missing_summary = missing_summary.sort_values('Jumlah Missing Value', ascending=False)

print('=== Tabel Jumlah Missing Value per Variabel ===')
print(missing_summary.to_string())

# Identifikasi variabel dengan missing value terbanyak
most_missing_col = missing_summary['Jumlah Missing Value'].idxmax()
most_missing_val = missing_summary['Jumlah Missing Value'].max()

print(f'\nVariabel dengan data hilang terbanyak: "{most_missing_col}" ({most_missing_val} nilai kosong)')

=== Tabel Jumlah Missing Value per Variabel ===
                   Jumlah Missing Value  Persentase (%)
normalized-losses                    41           20.00
jumlah-pintu                         18            8.78
bahan-bakar                          13            6.34
body-style                           12            5.85
length                                8            3.90
width                                 5            2.44
make                                  4            1.95
height                                4            1.95
wheel-bases                           3            1.46

Variabel dengan data hilang terbanyak: "normalized-losses" (41 nilai kosong)


**Penjelasan:**  
- Fungsi **isnull().sum()** digunakan untuk menghitung jumlah nilai **NaN** pada setiap kolom.  
- Variabel **normalized-losses** memiliki missing value terbanyak (41 nilai), diikuti oleh **jumlah-pintu** (18), **bahan-bakar** (13), dan **body-style** (12).  
- Data hilang perlu ditangani sebelum analisis atau pemodelan dilakukan agar hasil tidak bias.

---
## 2. Mengganti Missing Value pada Variabel Kategori
Untuk variabel berbentuk kategori, data yang hilang diganti dengan label **"unknown"**. Pendekatan ini mempertahankan jumlah baris data tanpa membuang informasi yang ada.

In [9]:
# Daftar variabel kategorik yang akan diisi dengan 'unknown'
categorical_cols = ['make', 'bahan-bakar', 'jumlah-pintu', 'body-style']

# Isi missing value pada setiap variabel kategorik dengan 'unknown'
# fillna() mengganti nilai NaN dengan nilai yang ditentukan
for col in categorical_cols:
    df[col] = df[col].fillna('unknown')

print('=== Tabel Ringkasan Jumlah Setiap Kategori ===')
print()

# Tampilkan distribusi nilai untuk setiap variabel kategorik
for col in categorical_cols:
    print(f'--- Variabel: {col} ---')
    value_counts = df[col].value_counts()
    print(value_counts.to_string())
    print()

# Konfirmasi: tidak ada lagi missing value di kolom kategorik
print('=== Konfirmasi Missing Value pada Variabel Kategorik Setelah Imputasi ===')
print(df[categorical_cols].isnull().sum())

=== Tabel Ringkasan Jumlah Setiap Kategori ===

--- Variabel: make ---
make
toyota           32
mazda            17
nissan           17
honda            13
mitsubishi       12
volkswagen       12
volvo            11
subaru           11
peugot           11
dodge             9
mercedes-benz     8
bmw               8
plymouth          7
audi              7
saab              6
porsche           4
unknown           4
isuzu             4
alfa-romero       3
chevrolet         3
jaguar            3
renault           2
mercury           1

--- Variabel: bahan-bakar ---
bahan-bakar
gas        173
diesel      19
unknown     13

--- Variabel: jumlah-pintu ---
jumlah-pintu
four       101
two         86
unknown     18

--- Variabel: body-style ---
body-style
sedan          88
hatchback      67
wagon          24
unknown        12
hardtop         8
convertible     6

=== Konfirmasi Missing Value pada Variabel Kategorik Setelah Imputasi ===
make            0
bahan-bakar     0
jumlah-pintu    0
body-sty

**Penjelasan:**  
- Fungsi **fillna('unknown')** digunakan untuk mengisi nilai kosong dengan string **'unknown'**.  
- Strategi ini membuat kategori baru bernama **'unknown'** yang merepresentasikan data yang tidak diketahui.  
- Pendekatan ini lebih baik daripada menghapus baris karena tidak mengurangi jumlah data.

---
## 3. Mengganti Missing Value pada Variabel Numerik
Untuk variabel numerik yang memiliki data hilang, dilakukan **imputasi menggunakan nilai rata-rata (mean)**. Metode ini memiliki efek negatif yang minimum terhadap nilai rata-rata keseluruhan data.

In [10]:
# Daftar variabel numerik yang akan diisi dengan nilai rata-rata
numeric_impute_cols = ['length', 'width', 'height']

print('=== Nilai Rata-Rata (Mean) Sebelum Imputasi ===')
for col in numeric_impute_cols:
    mean_val = df[col].mean()
    missing_count_before = df[col].isnull().sum()
    print(f'  {col}: mean = {mean_val:.4f} | missing = {missing_count_before}')

print()

# Isi missing value pada variabel numerik dengan nilai rata-rata masing-masing
# mean() menghitung rata-rata dari semua nilai yang tidak kosong
for col in numeric_impute_cols:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)

print('=== Konfirmasi Missing Value pada Variabel Numerik Setelah Imputasi ===')
print(df[numeric_impute_cols].isnull().sum())

print()
print('=== Statistik Deskriptif Variabel Numerik Setelah Imputasi ===')
print(df[numeric_impute_cols].describe().round(4))

=== Nilai Rata-Rata (Mean) Sebelum Imputasi ===
  length: mean = 173.9025 | missing = 8
  width: mean = 65.9310 | missing = 5
  height: mean = 53.7373 | missing = 4

=== Konfirmasi Missing Value pada Variabel Numerik Setelah Imputasi ===
length    0
width     0
height    0
dtype: int64

=== Statistik Deskriptif Variabel Numerik Setelah Imputasi ===
         length     width    height
count  205.0000  205.0000  205.0000
mean   173.9025   65.9310   53.7373
std     11.9072    2.1103    2.4225
min    141.1000   60.3000   47.8000
25%    167.3000   64.2000   52.0000
50%    173.4000   65.5000   54.1000
75%    180.2000   66.6000   55.5000
max    208.1000   72.3000   59.8000


**Penjelasan:**  
- Fungsi **mean()** menghitung nilai rata-rata dari semua data yang tersedia (mengabaikan NaN secara otomatis).  
- Fungsi **fillna(mean_val)** kemudian mengisi setiap nilai kosong dengan rata-rata tersebut.  
- Imputasi dengan mean tidak mengubah nilai rata-rata keseluruhan, sehingga distribusi data relatif tetap terjaga.

---
## 4. Min-Max Normalization pada Variabel **wheel-bases**
Min-Max Normalization mengubah skala data ke rentang **[0, 1]**. Rumus yang digunakan:  

$$v' = \frac{v - min_v}{max_v - min_v}$$

Metode ini berguna agar variabel dengan rentang nilai besar tidak mendominasi analisis berbasis jarak.

In [11]:
# Isi missing value pada wheel-bases dengan mean sebelum normalisasi
# (wheel-bases memiliki 3 missing value berdasarkan eksplorasi awal)
df['wheel-bases'] = df['wheel-bases'].fillna(df['wheel-bases'].mean())

# Hitung nilai minimum dan maksimum dari variabel wheel-bases
min_wb = df['wheel-bases'].min()
max_wb = df['wheel-bases'].max()

print(f'Nilai Minimum wheel-bases : {min_wb}')
print(f'Nilai Maksimum wheel-bases: {max_wb}')
print()

# Terapkan rumus Min-Max Normalization
# v' = (v - min) / (max - min)
df['wheel-bases-normalized'] = (df['wheel-bases'] - min_wb) / (max_wb - min_wb)

print('=== Perbandingan nilai wheel-bases sebelum dan sesudah normalisasi ===')
print(df[['wheel-bases', 'wheel-bases-normalized']].head(10).to_string())

print()
print('=== Statistik Deskriptif wheel-bases-normalized ===')
print(df['wheel-bases-normalized'].describe().round(6))
print(f'\nNilai minimum hasil normalisasi : {df["wheel-bases-normalized"].min()}')
print(f'Nilai maksimum hasil normalisasi: {df["wheel-bases-normalized"].max()}')

Nilai Minimum wheel-bases : 86.6
Nilai Maksimum wheel-bases: 120.9

=== Perbandingan nilai wheel-bases sebelum dan sesudah normalisasi ===
   wheel-bases  wheel-bases-normalized
0         88.6                0.058309
1         88.6                0.058309
2         94.5                0.230321
3         99.8                0.384840
4         99.4                0.373178
5         99.8                0.384840
6        105.8                0.559767
7        105.8                0.559767
8        105.8                0.559767
9         99.5                0.376093

=== Statistik Deskriptif wheel-bases-normalized ===
count    205.000000
mean       0.355209
std        0.175216
min        0.000000
25%        0.230321
50%        0.303207
75%        0.460641
max        1.000000
Name: wheel-bases-normalized, dtype: float64

Nilai minimum hasil normalisasi : 0.0
Nilai maksimum hasil normalisasi: 1.0


**Penjelasan:**  
- Min-Max Normalization memetakan setiap nilai ke dalam rentang [0, 1].  
- Nilai minimum asli akan menjadi 0, dan nilai maksimum asli akan menjadi 1.  
- Kolom baru **wheel-bases-normalized** menyimpan hasil normalisasi tanpa mengubah data asli.

---
## 5. Z-Score Normalization pada    Variabel **length**, **width**, **height**
Z-Score Normalization (Standardisasi) mengubah data sehingga memiliki **rata-rata = 0** dan **standar deviasi = 1**. Rumus yang digunakan:  

$$v' = \frac{v - \bar{v}}{\sigma_v}$$

Metode ini berguna ketika data tidak terdistribusi dalam rentang tertentu dan cocok untuk algoritma yang sensitif terhadap skala data.

In [12]:
# Variabel yang akan dinormalisasi dengan Z-Score
zscore_cols = ['length', 'width', 'height']

print('=== Statistik Sebelum Z-Score Normalization ===')
print(df[zscore_cols].describe().round(4))
print()

# Terapkan Z-Score Normalization untuk setiap variabel
# v' = (v - mean) / std
for col in zscore_cols:
    mean_val = df[col].mean()    # Rata-rata variabel
    std_val  = df[col].std()     # Standar deviasi variabel

    # Buat kolom baru dengan nama {kolom}_zscore
    df[f'{col}_zscore'] = (df[col] - mean_val) / std_val

    print(f'  {col}: mean = {mean_val:.4f}, std = {std_val:.4f}')

print()
zscore_result_cols = ['length_zscore', 'width_zscore', 'height_zscore']

print('=== Statistik Setelah Z-Score Normalization ===')
print(df[zscore_result_cols].describe().round(6))

print()
print('=== Perbandingan Nilai Asli vs Z-Score (10 baris pertama) ===')
comparison_cols = ['length', 'length_zscore', 'width', 'width_zscore', 'height', 'height_zscore']
print(df[comparison_cols].head(10).round(4).to_string())

=== Statistik Sebelum Z-Score Normalization ===
         length     width    height
count  205.0000  205.0000  205.0000
mean   173.9025   65.9310   53.7373
std     11.9072    2.1103    2.4225
min    141.1000   60.3000   47.8000
25%    167.3000   64.2000   52.0000
50%    173.4000   65.5000   54.1000
75%    180.2000   66.6000   55.5000
max    208.1000   72.3000   59.8000

  length: mean = 173.9025, std = 11.9072
  width: mean = 65.9310, std = 2.1103
  height: mean = 53.7373, std = 2.4225

=== Statistik Setelah Z-Score Normalization ===
       length_zscore  width_zscore  height_zscore
count     205.000000    205.000000     205.000000
mean        0.000000      0.000000       0.000000
std         1.000000      1.000000       1.000000
min        -2.754853     -2.668281      -2.450888
25%        -0.554500     -0.820244      -0.717153
50%        -0.042205     -0.204232       0.149715
75%         0.528879      0.317009       0.727627
max         2.872003      3.017986       2.502641

=== Perba

**Penjelasan:**  
- **mean()** menghitung rata-rata dan **std()** menghitung standar deviasi dari setiap variabel.  
- Setelah Z-Score normalization, rata-rata kolom hasil mendekati **0** dan standar deviasi mendekati **1**.  
- Kolom baru dengan suffix **_zscore** dibuat agar data asli tetap tersimpan untuk perbandingan.

---
## Ringkasan Dataset Akhir

In [13]:
# Tampilkan ringkasan dataset setelah semua proses data preparation
print('=== Kolom Dataset Setelah Data Preparation ===')
print(df.columns.tolist())
print(f'\nTotal baris  : {df.shape[0]}')
print(f'Total kolom  : {df.shape[1]}')

print()
print('=== Konfirmasi: Sisa Missing Value di Seluruh Dataset ===')
remaining_missing = df.isnull().sum()
print(remaining_missing[remaining_missing > 0] if remaining_missing.sum() > 0 else 'Tidak ada missing value yang tersisa (kecuali normalized-losses yang tidak ditangani).')

print()
print('=== 5 Baris Pertama Dataset Final ===')
display_cols = ['make', 'bahan-bakar', 'jumlah-pintu', 'body-style',
                'wheel-bases', 'wheel-bases-normalized',
                'length', 'length_zscore',
                'width', 'width_zscore',
                'height', 'height_zscore']
print(df[display_cols].head().round(4).to_string())

=== Kolom Dataset Setelah Data Preparation ===
['normalized-losses', 'make', 'bahan-bakar', 'jumlah-pintu', 'body-style', 'wheel-bases', 'length', 'width', 'height', 'wheel-bases-normalized', 'length_zscore', 'width_zscore', 'height_zscore']

Total baris  : 205
Total kolom  : 13

=== Konfirmasi: Sisa Missing Value di Seluruh Dataset ===
normalized-losses    41
dtype: int64

=== 5 Baris Pertama Dataset Final ===
          make bahan-bakar jumlah-pintu   body-style  wheel-bases  wheel-bases-normalized  length  length_zscore  width  width_zscore  height  height_zscore
0  alfa-romero         gas          two  convertible         88.6                  0.0583   168.8        -0.4285   64.1       -0.8676    48.8        -2.0381
1  alfa-romero         gas          two  convertible         88.6                  0.0583   168.8        -0.4285   64.1       -0.8676    48.8        -2.0381
2  alfa-romero         gas          two    hatchback         94.5                  0.2303   171.2        -0.2270  